# PAQFE-v2 — Trainable Physics Scaling

**Experiment 1 only.** This notebook changes one thing in the current PAQFE: fixed physics scaling becomes trainable physics scaling. Everything else is kept unchanged.

Current: `theta_i = fixed_scale_i * x_i`

PAQFE-v2: `theta_i = alpha_i * x_i`, with `alpha` initialized from the existing physics prior and softly regularized toward that prior.


In [ ]:
!pip install pennylane

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pennylane as qml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)


In [ ]:


GROUPED_FEATURE_ORDER = ["xAl","VEC","dchi","dHmix","dSmix","Omega","Mean_Tm","delta","Temperature_C","log10_Strain_rate"]
TARGET = "UTS_MPa"
STRONG_RELATIONSHIP_DESCRIPTORS = {"delta","Omega","xAl","VEC"}
UPWEIGHT_FACTOR = 1.2
N_QUBITS = 10
N_LAYERS = 3
LR = 0.02
N_EPOCHS = 300
LAMBDA_PRIOR = 0.01
composition_cols = ["Al_norm","Co_norm","Cr_norm","Fe_norm","Ni_norm"]


In [ ]:
from google.colab import files

uploaded = files.upload()
file_path = list(uploaded.keys())[0]

df = pd.read_excel(
    file_path,
    sheet_name="B_Model_Ready"
)

print("Dataset shape:", df.shape)
display(df.head())
cols = GROUPED_FEATURE_ORDER + [TARGET] + composition_cols
df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")
df = df.dropna(subset=cols).copy()
df["Composition_Group"] = df[composition_cols].round(3).astype(str).agg("_".join, axis=1)
print("Observations:", len(df))
print("Unique compositions:", df["Composition_Group"].nunique())


Saving PAQFE_UTS_Dataset_Corrected (1).xlsx to PAQFE_UTS_Dataset_Corrected (1) (3).xlsx
Dataset shape: (49, 25)


,Observation_ID,Al,Co,Cr,Fe,Ni,Al_norm,Co_norm,Cr_norm,Fe_norm,...,delta,Omega,dchi,Mean_Tm,OSHE,NVC,Temperature_C,Strain_rate,log10_Strain_rate,UTS_MPa
0,OBS_001,0.000,0.250,0.250,0.250,0.250,0.000000,0.250000,0.250000,0.250000,...,1.103880,5.752837,0.096695,1871.750000,NaN,NaN,900,0.001,-3.0,130.0
1,OBS_002,0.048,0.238,0.238,0.238,0.238,0.048000,0.238000,0.238000,0.238000,...,3.076890,3.685665,0.104482,1826.714000,NaN,NaN,900,0.001,-3.0,148.0
2,OBS_003,0.111,0.222,0.222,0.222,0.222,0.111111,0.222222,0.222222,0.222222,...,4.328494,2.557021,0.112546,1767.500000,NaN,NaN,900,0.001,-3.0,172.0
3,OBS_004,0.130,0.217,0.217,0.217,0.217,0.130261,0.217435,0.217435,0.217435,...,4.604955,2.349226,0.114579,1749.533066,NaN,NaN,900,0.001,-3.0,188.0
4,OBS_005,0.200,0.200,0.200,0.200,0.200,0.200000,0.200000,0.200000,0.200000,...,5.376204,1.829117,0.120565,1684.100000,NaN,NaN,900,0.001,-3.0,215.0


Observations: 49
Unique compositions: 14


In [ ]:
TEST_SIZE = 0.20
SPLIT_RANDOM_STATE = 42
train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=SPLIT_RANDOM_STATE)
X_train_raw = train_df[GROUPED_FEATURE_ORDER].to_numpy(dtype=np.float64)
X_test_raw = test_df[GROUPED_FEATURE_ORDER].to_numpy(dtype=np.float64)
y_train_raw = train_df[TARGET].to_numpy(dtype=np.float64)
y_test_raw = test_df[TARGET].to_numpy(dtype=np.float64)
x_scaler = StandardScaler()
X_train = x_scaler.fit_transform(X_train_raw)
X_test = x_scaler.transform(X_test_raw)
y_scaler = StandardScaler()
y_train = y_scaler.fit_transform(y_train_raw.reshape(-1,1)).ravel()


In [ ]:
QUBIT_BLOCKS = {"composition":[0,1,2], "thermodynamic":[3,4,5,6], "structural":[7], "context":[8,9]}
def ring_entangle(wires):
    if len(wires) < 2: return
    for i in range(len(wires)): qml.CNOT(wires=[wires[i], wires[(i+1)%len(wires)]])
def physics_entangling_layer():
    ring_entangle(QUBIT_BLOCKS["composition"])
    ring_entangle(QUBIT_BLOCKS["thermodynamic"])
    ring_entangle(QUBIT_BLOCKS["context"])
    fmap = {f:i for i,f in enumerate(GROUPED_FEATURE_ORDER)}
    qml.CNOT(wires=[fmap["delta"], fmap["Omega"]])
    qml.CNOT(wires=[fmap["xAl"], fmap["VEC"]])


In [ ]:
physics_prior = torch.tensor([UPWEIGHT_FACTOR if f in STRONG_RELATIONSHIP_DESCRIPTORS else 1.0 for f in GROUPED_FEATURE_ORDER], dtype=torch.float64)
dev = qml.device("default.qubit", wires=N_QUBITS)
@qml.qnode(dev, interface="torch", diff_method="backprop")
def circuit(inputs, weights):
    qml.templates.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation="Y")
    physics_entangling_layer()
    qml.templates.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]
weight_shape = qml.StronglyEntanglingLayers.shape(n_layers=N_LAYERS, n_wires=N_QUBITS)
class PAQFEv2(nn.Module):
    def __init__(self):
        super().__init__()
        self.alpha = nn.Parameter(physics_prior.clone())
        self.weights = nn.Parameter(0.01 * torch.randn(weight_shape, dtype=torch.float64))
        self.head = nn.Linear(N_QUBITS, 1, dtype=torch.float64)
    def forward(self, x):
        theta = x * self.alpha
        q = torch.stack(circuit(theta, self.weights), dim=1)
        return self.head(q).squeeze(-1)
model = PAQFEv2()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.MSELoss()


In [ ]:
Xtr = torch.tensor(X_train, dtype=torch.float64)
ytr = torch.tensor(y_train, dtype=torch.float64)
loss_history=[]
for epoch in range(N_EPOCHS):
    optimizer.zero_grad()
    pred=model(Xtr)
    mse=loss_fn(pred,ytr)
    prior_penalty=torch.mean((model.alpha-physics_prior)**2)
    loss=mse + LAMBDA_PRIOR*prior_penalty
    loss.backward()
    optimizer.step()
    loss_history.append(float(loss.detach()))
print("Final loss:", loss_history[-1])


Final loss: 0.04672516081694967


In [ ]:
model.eval()
with torch.no_grad(): pred_std=model(torch.tensor(X_test,dtype=torch.float64)).cpu().numpy()
pred=y_scaler.inverse_transform(pred_std.reshape(-1,1)).ravel()
mae=mean_absolute_error(y_test_raw,pred)
rmse=np.sqrt(mean_squared_error(y_test_raw,pred))
r2=r2_score(y_test_raw,pred)
print(f"PAQFE-v2 MAE  = {mae:.3f}")
print(f"PAQFE-v2 RMSE = {rmse:.3f}")
print(f"PAQFE-v2 R2   = {r2:.3f}")


PAQFE-v2 MAE  = 205.371
PAQFE-v2 RMSE = 256.873
PAQFE-v2 R2   = 0.318


In [ ]:
alpha_final=model.alpha.detach().cpu().numpy()
alpha_table=pd.DataFrame({"Feature":GROUPED_FEATURE_ORDER,"Physics_prior":physics_prior.numpy(),"Learned_alpha":alpha_final,"Change":alpha_final-physics_prior.numpy()})
display(alpha_table)
results=pd.DataFrame({"Model":["PAQFE-v2_trainable_physics_scaling"],"MAE":[mae],"RMSE":[rmse],"R2":[r2],"Seed":[SEED],"Lambda_prior":[LAMBDA_PRIOR]})
display(results)
results.to_csv("PAQFE_v2_trainable_scaling_result.csv",index=False)
alpha_table.to_csv("PAQFE_v2_learned_alpha.csv",index=False)


,Feature,Physics_prior,Learned_alpha,Change
0,xAl,1.2,0.845992,-0.354008
1,VEC,1.2,1.041673,-0.158327
2,dchi,1.0,0.708564,-0.291436
3,dHmix,1.0,0.817355,-0.182645
4,dSmix,1.0,0.857601,-0.142399
5,Omega,1.2,0.838241,-0.361759
6,Mean_Tm,1.0,0.608463,-0.391537
7,delta,1.2,0.912099,-0.287901
8,Temperature_C,1.0,1.209282,0.209282
9,log10_Strain_rate,1.0,1.228265,0.228265


,Model,MAE,RMSE,R2,Seed,Lambda_prior
0,PAQFE-v2_trainable_physics_scaling,205.371255,256.872717,0.317792,42,0.01


In [ ]:
# ============================================================
# PAQFE-v2 STABILITY CHECK — SEEDS 42–46
# Trainable alpha uses lr=0.005
# VQC weights + regression head use lr=0.02
# ============================================================

SEEDS = [42, 43, 44, 45, 46]
seed_results = []

for seed in SEEDS:

    # Reproducible initialization
    np.random.seed(seed)
    torch.manual_seed(seed)

    # Fresh PAQFE-v2 model for this seed
    model_seed = PAQFEv2()

    # Separate learning rates:
    # alpha = 0.005
    # quantum weights = 0.02
    # regression head = 0.02
    optimizer_seed = torch.optim.Adam([
        {"params": model_seed.alpha, "lr": 0.005},
        {"params": model_seed.weights, "lr": 0.02},
        {"params": model_seed.head.parameters(), "lr": 0.02}
    ])

    loss_fn_seed = nn.MSELoss()

    # Same training data
    Xtr_seed = torch.tensor(X_train, dtype=torch.float64)
    ytr_seed = torch.tensor(y_train, dtype=torch.float64)

    # Train
    model_seed.train()

    for epoch in range(N_EPOCHS):

        optimizer_seed.zero_grad()

        pred = model_seed(Xtr_seed)

        mse = loss_fn_seed(pred, ytr_seed)

        prior_penalty = torch.mean(
            (model_seed.alpha - physics_prior) ** 2
        )

        loss = mse + LAMBDA_PRIOR * prior_penalty

        loss.backward()
        optimizer_seed.step()

    # Test
    model_seed.eval()

    with torch.no_grad():

        pred_std_seed = model_seed(
            torch.tensor(X_test, dtype=torch.float64)
        ).cpu().numpy()

    pred_seed = y_scaler.inverse_transform(
        pred_std_seed.reshape(-1, 1)
    ).ravel()

    # Metrics
    mae_seed = mean_absolute_error(
        y_test_raw, pred_seed
    )

    rmse_seed = np.sqrt(
        mean_squared_error(y_test_raw, pred_seed)
    )

    r2_seed = r2_score(
        y_test_raw, pred_seed
    )

    seed_results.append({
        "Seed": seed,
        "MAE": mae_seed,
        "RMSE": rmse_seed,
        "R2": r2_seed
    })

    print(
        f"Seed {seed}: "
        f"MAE={mae_seed:.3f}, "
        f"RMSE={rmse_seed:.3f}, "
        f"R2={r2_seed:.3f}"
    )


# ============================================================
# Results
# ============================================================

seed_results_df = pd.DataFrame(seed_results)

print("\n===== PAQFE-v2 Stability Results =====")
display(seed_results_df)

summary = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2"],
    "Mean": [
        seed_results_df["MAE"].mean(),
        seed_results_df["RMSE"].mean(),
        seed_results_df["R2"].mean()
    ],
    "SD": [
        seed_results_df["MAE"].std(ddof=1),
        seed_results_df["RMSE"].std(ddof=1),
        seed_results_df["R2"].std(ddof=1)
    ]
})

print("\n===== Mean ± SD =====")
display(summary)

# Save
seed_results_df.to_csv(
    "PAQFE_v2_slow_alpha_seed_stability_results.csv",
    index=False
)

summary.to_csv(
    "PAQFE_v2_slow_alpha_seed_stability_summary.csv",
    index=False
)

Seed 42: MAE=203.443, RMSE=253.647, R2=0.335
Seed 43: MAE=230.581, RMSE=292.788, R2=0.114
Seed 44: MAE=227.631, RMSE=288.297, R2=0.141
Seed 45: MAE=205.293, RMSE=255.469, R2=0.325
Seed 46: MAE=193.110, RMSE=242.180, R2=0.394

===== PAQFE-v2 Stability Results =====


,Seed,MAE,RMSE,R2
0,42,203.443030,253.646977,0.334818
1,43,230.580515,292.787558,0.113689
2,44,227.630733,288.296744,0.140669
3,45,205.292953,255.469283,0.325226
4,46,193.110018,242.179922,0.393603



===== Mean ± SD =====


,Metric,Mean,SD
0,MAE,212.011450,16.314045
1,RMSE,266.476097,22.607930
2,R2,0.261601,0.125833


## Stop here

Compare this result against the existing PAQFE result. Do **not** change the entanglement, depth, learning rate, lambda, or split yet. If this experiment improves, we keep it as PAQFE-v2 and move to the next controlled experiment. If it does not, we isolate the physics-entanglement component next.